<style>
.jp-Notebook { max-width: 1000px; margin: 0 auto; }
.jp-RenderedMarkdown h1 { border-bottom: 2px solid #ddd; padding-bottom: .3em; }
.jp-RenderedMarkdown h2 { margin-top: 1.8em; color: #123; border-bottom: 1px solid #eee; }
.jp-RenderedMarkdown h3 { margin-top: 1.4em; color: #345; }
.jp-RenderedHTMLCommon table { margin: 1em 0; }
</style>

# Exercício 1 — Nuvens de Pontos: Geometria e Dispersão em 2D

**Redes Neurais — Insper, 2026.2**

Entender como os dados estão distribuídos é o primeiro passo antes de projetar uma arquitetura
de rede. Neste exercício vamos **gerar** e **medir** nuvens de pontos bidimensionais,
observando como a distribuição afeta a complexidade das fronteiras de decisão que uma rede
neural precisaria aprender.

**Conformidade com as regras técnicas.** Semente fixa via `rng = np.random.default_rng(42)`,
com **um único objeto `rng` reutilizado em todo o relatório** — o notebook é executado de cima
para baixo (`jupyter nbconvert --execute`), então a sequência de sorteios e todos os números
desta página são reprodutíveis. Bibliotecas: **NumPy**, **pandas**, **Matplotlib** e **scikit-learn** — este último apenas
para o `PCA` do Exercício 2 e para o pré-processamento do Exercício 3 (imputação, one-hot,
scaling e split). **Nenhum modelo é treinado** em nenhum dos três exercícios: as medidas dos
Exercícios 1 e 2 são geométricas, e o Exercício 3 para na matriz pronta para a rede.

**Uso de ferramentas de IA.** Este relatório foi produzido com apoio do Claude (Anthropic) na
escrita do código, na geração das figuras e na redação das análises. As decisões metodológicas,
os parâmetros e as interpretações foram revisados e validados por mim, e consigo explicar cada
trecho do código e cada conclusão apresentada. Todo número pedido pelo
enunciado está escrito no texto, além de aparecer na saída do código.

**Configuração**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (7, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

# ÚNICO gerador aleatório do relatório: criado aqui e reutilizado em todas as
# amostragens abaixo, na ordem em que as células aparecem.
rng = np.random.default_rng(42)

N_PER_CLASS = 100  # 100 amostras por classe -> 400 no total
N_CLASSES = 4

# Parâmetros do enunciado: uma linha por classe
PARAMS = pd.DataFrame(
    {
        "mu_x":    [2.0, 5.0, 8.0, 15.0],
        "mu_y":    [3.0, 6.0, 1.0,  4.0],
        "sigma_x": [0.8, 1.2, 0.9,  0.5],
        "sigma_y": [2.5, 1.9, 0.9,  2.0],
    },
    index=pd.Index(range(N_CLASSES), name="classe"),
)

MEANS = PARAMS[["mu_x", "mu_y"]].to_numpy()        # (4, 2)
STDS  = PARAMS[["sigma_x", "sigma_y"]].to_numpy()  # (4, 2)

COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"]

PARAMS

In [ ]:
def gerar_nuvens(scale=1.0, n=N_PER_CLASS):
    """Amostra as 4 nuvens gaussianas com os desvios multiplicados por `scale`.

    Usa o `rng` global do relatório (não cria um novo), de modo que sorteios
    sucessivos avancem o mesmo fluxo aleatório. As médias nunca mudam; apenas a
    dispersão. Retorna (X, y) com X de shape (4*n, 2) e y de shape (4*n,).
    """
    X = np.concatenate([
        rng.normal(loc=MEANS[k], scale=STDS[k] * scale, size=(n, 2))
        for k in range(N_CLASSES)
    ])
    y = np.repeat(np.arange(N_CLASSES), n)
    return X, y


def plotar_nuvens(X, y, ax, marcar_centros=True, tamanho=14):
    """Desenha as nuvens coloridas por classe; opcionalmente marca as médias."""
    for k in range(N_CLASSES):
        pts = X[y == k]
        ax.scatter(pts[:, 0], pts[:, 1], s=tamanho, alpha=0.65,
                   color=COLORS[k], edgecolors="none", label=f"Classe {k}")
    if marcar_centros:
        ax.scatter(MEANS[:, 0], MEANS[:, 1], marker="X", s=170,
                   c=COLORS, edgecolors="black", linewidths=1.4,
                   zorder=5, label="Centro (média)")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")


def taxa_de_mistura(X, y, means=MEANS):
    """Fração de pontos cujo centro de classe mais próximo não é o da própria classe.

    Medida puramente geométrica: compara cada ponto contra as 4 médias por
    broadcasting. Nada é treinado. Retorna (taxa, indice_do_centro_mais_proximo).
    """
    dists = np.linalg.norm(X[:, None, :] - means[None, :, :], axis=2)  # (n_pontos, 4)
    mais_proximo = dists.argmin(axis=1)
    return (mais_proximo != y).mean(), mais_proximo

## A — Gerar as nuvens

Dataset sintético com 400 amostras no total, divididas igualmente entre 4 classes
(100 amostras cada), com distribuição gaussiana e os parâmetros do enunciado:

| Classe | Média | Desvio-padrão |
|:------:|:-----:|:-------------:|
| 0 | $[2,\ 3]$   | $[0{,}8,\ 2{,}5]$ |
| 1 | $[5,\ 6]$   | $[1{,}2,\ 1{,}9]$ |
| 2 | $[8,\ 1]$   | $[0{,}9,\ 0{,}9]$ |
| 3 | $[15,\ 4]$  | $[0{,}5,\ 2{,}0]$ |

In [ ]:
X, y = gerar_nuvens(scale=1.0)   # primeiro sorteio do rng

print(f"X.shape = {X.shape}   y.shape = {y.shape}")
print("amostras por classe:", np.bincount(y))
print("valores ausentes:", int(np.isnan(X).sum()))

In [ ]:
# Estatísticas empíricas x parâmetros teóricos (pandas)
df = pd.DataFrame(X, columns=["x1", "x2"]).assign(classe=y)

resumo_A = df.groupby("classe").agg(
    media_x1=("x1", "mean"), media_x2=("x2", "mean"),
    desvio_x1=("x1", "std"), desvio_x2=("x2", "std"),
)
resumo_A.insert(0, "mu_x_teorico", PARAMS["mu_x"])
resumo_A.insert(2, "mu_y_teorico", PARAMS["mu_y"])

resumo_A.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
plotar_nuvens(X, y, ax)

# rótulo textual em cada centro (deslocamentos escolhidos para não cortar)
OFFSETS = [(14, -22), (14, 12), (14, 12), (-96, 14)]
for k in range(N_CLASSES):
    ax.annotate(f"$\\mu_{k}$ = ({MEANS[k,0]:.0f}, {MEANS[k,1]:.0f})",
                MEANS[k], textcoords="offset points", xytext=OFFSETS[k],
                fontsize=9, fontweight="bold", color=COLORS[k],
                bbox=dict(boxstyle="round,pad=0.25", fc="white",
                          ec=COLORS[k], lw=0.8, alpha=0.9))
ax.margins(0.08)

ax.set_title("Figura 1 — Nuvens de pontos 2D ($s = 1$), com os centros marcados")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend(title="Classe", loc="upper left", framealpha=0.9, fontsize=9)
fig.tight_layout()
plt.show()

**Leitura da Figura 1.** As 400 amostras se distribuem em 100 por classe, sem valores
ausentes. As médias empíricas reproduzem bem os parâmetros teóricos — por exemplo, a classe 0
saiu com centro em $(1{,}988,\ 2{,}885)$ contra $[2,\ 3]$ teórico, e a classe 3 em
$(14{,}993,\ 3{,}881)$ contra $[15,\ 4]$.

Geometricamente: as classes 0 e 1 estão próximas e ambas são bastante alongadas no eixo $x_2$
(desvios 2,5 e 1,9) — elas se tocam. A classe 2 é a mais compacta (desvio 0,9 nos dois eixos)
e fica isolada abaixo. A classe 3 está longe no eixo $x_1$ ($\mu_x = 15$), estreita em $x_1$
e esticada em $x_2$, formando uma faixa vertical completamente separada das demais.

## B — Mais ou menos dispersas

Geramos as **mesmas 4 classes do item A** quatro vezes, multiplicando **todos** os
desvios-padrão por um fator de escala $s$ — as médias nunca mudam, só a dispersão:

$$s \in \{0{,}5,\ 1{,}0,\ 2{,}0,\ 4{,}0\}$$

São 4 datasets de 4 classes cada. O dataset de $s = 1$ é exatamente o do item A (reaproveitado,
não sorteado de novo); os outros três são sorteados na ordem $0{,}5 \to 2{,}0 \to 4{,}0$,
avançando o mesmo `rng`.

### B.1 — Figura 2: os quatro datasets com eixos compartilhados

In [ ]:
SCALES = [0.5, 1.0, 2.0, 4.0]

datasets = {}
for s in SCALES:
    if s == 1.0:
        datasets[s] = (X, y)          # o dataset do item A, sem novo sorteio
    else:
        datasets[s] = gerar_nuvens(scale=s)

# Limites de eixo comuns a TODOS os datasets, para a comparação ser honesta
todos = np.concatenate([Xs for Xs, _ in datasets.values()])
folga = 1.0
XLIM = (todos[:, 0].min() - folga, todos[:, 0].max() + folga)
YLIM = (todos[:, 1].min() - folga, todos[:, 1].max() + folga)

print(f"limites compartilhados: x em ({XLIM[0]:.1f}, {XLIM[1]:.1f}), "
      f"y em ({YLIM[0]:.1f}, {YLIM[1]:.1f})")
for s in SCALES:
    print(f"  s = {s}: {datasets[s][0].shape[0]} pontos")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8.5), sharex=True, sharey=True)

for ax, s in zip(axes.ravel(), SCALES):
    Xs, ys = datasets[s]
    plotar_nuvens(Xs, ys, ax, tamanho=10)
    ax.set_title(f"Dispersão escalada: $s = {s}$", fontsize=10)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)
    ax.legend(title="Classe", loc="upper left", fontsize=7,
              title_fontsize=7, framealpha=0.9)

fig.suptitle("Figura 2 — Mesmas 4 classes com dispersão escalada por $s$ "
             "(eixos compartilhados)", fontsize=13)
fig.tight_layout()
plt.show()

Com os eixos travados nos mesmos limites o efeito fica visível: em $s = 0{,}5$ as quatro
nuvens são pontos compactos e bem separados; em $s = 4$ elas cobrem quase todo o plano e se
sobrepõem umas às outras.

### B.2 — Razão de separação (para $s = 1$)

Para cada par de classes $(i, j)$:

$$r_{ij} = \frac{\lVert \mu_i - \mu_j \rVert}{\bar{\sigma}_i + \bar{\sigma}_j},
\qquad
\bar{\sigma}_k = \frac{\sigma_{k,x} + \sigma_{k,y}}{2}$$

Valores altos significam nuvens bem separadas; valores baixos, nuvens que se misturam.

In [ ]:
sigma_bar = STDS.mean(axis=1)              # sigma médio de cada classe
print("sigma_bar =", dict(zip(range(N_CLASSES), sigma_bar.round(3))))

linhas = []
for i in range(N_CLASSES):
    for j in range(i + 1, N_CLASSES):
        dist = np.linalg.norm(MEANS[i] - MEANS[j])
        linhas.append({
            "par": f"({i}, {j})",
            "||mu_i - mu_j||": dist,
            "sigma_bar_i + sigma_bar_j": sigma_bar[i] + sigma_bar[j],
            "r_ij": dist / (sigma_bar[i] + sigma_bar[j]),
        })

razoes = pd.DataFrame(linhas).set_index("par").sort_values("r_ij")
razoes.round(4)

In [ ]:
par_min = razoes["r_ij"].idxmin()
r_min = razoes["r_ij"].min()

print(f"Menor razão de separação: par {par_min}  ->  r = {r_min:.4f}")
print(f"Maior razão de separação: par {razoes['r_ij'].idxmax()}  ->  "
      f"r = {razoes['r_ij'].max():.4f}")
print(f"Como r_ij escala com 1/s e as médias não mudam, em s = 2 esse menor valor vira "
      f"{r_min:.4f} / 2 = {r_min / 2:.4f}  (sem gerar nada de novo)")

**Resposta B.2.** São 6 pares, todos na tabela acima. O **menor** é o par **(0, 1)**, com
$r_{01} = \mathbf{1{,}3258}$: os centros $[2, 3]$ e $[5, 6]$ distam apenas
$\lVert[3, 3]\rVert = 4{,}2426$, enquanto a soma dos espalhamentos médios é
$\bar{\sigma}_0 + \bar{\sigma}_1 = 1{,}65 + 1{,}55 = 3{,}20$ — praticamente do mesmo tamanho.
Os demais pares, em ordem: $r_{12} = 2{,}3800$, $r_{02} = 2{,}4802$, $r_{23} = 3{,}5422$,
$r_{13} = 3{,}6422$ e, o maior, $r_{03} = 4{,}4960$ — todos confortavelmente separados.

Como o numerador $\lVert \mu_i - \mu_j \rVert$ não depende de $s$ e o denominador é
proporcional a $s$, vale $r_{ij}(s) = r_{ij}(1)/s$. Logo, em $s = 2$ o menor valor passa a
$r_{01} = 1{,}3258 / 2 = \mathbf{0{,}6629}$ — **abaixo de 1**, ou seja, a distância entre os
centros fica *menor* que a soma dos espalhamentos: as duas nuvens deixam de ser distinguíveis.

### B.3 — Taxa de mistura

Para cada $s$, a fração de pontos cujo centro de classe **mais próximo** não é o da sua
própria classe. É uma medida puramente geométrica — comparamos cada ponto contra as 4 médias
com NumPy, sem nada a treinar.

In [ ]:
registros = []
for s in SCALES:
    Xs, ys = datasets[s]
    taxa, prox = taxa_de_mistura(Xs, ys)
    reg = {"s": s, "taxa_de_mistura": taxa,
           "pontos_misturados": int((prox != ys).sum()),
           "r_01": r_min / s}
    for k in range(N_CLASSES):
        reg[f"erro_classe_{k}"] = (prox[ys == k] != k).mean()
    registros.append(reg)

mistura = pd.DataFrame(registros).set_index("s")

for s, linha in mistura.iterrows():
    print(f"s = {s}: taxa de mistura = {linha['taxa_de_mistura']*100:.2f}% "
          f"({int(linha['pontos_misturados'])}/400 pontos), r_01 = {linha['r_01']:.4f}")

mistura.round(4)

**Resposta B.3 — as taxas de mistura.** Com o `rng` compartilhado, os quatro datasets dão:

| $s$ | taxa de mistura | pontos misturados |
|:---:|:---:|:---:|
| 0,5 | **0,00%** | 0 / 400 |
| 1,0 | **5,00%** | 20 / 400 |
| 2,0 | **19,25%** | 77 / 400 |
| 4,0 | **48,25%** | 193 / 400 |

### B.4 — Figura 3: taxa de mistura × fator de escala

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))

ax.plot(mistura.index, mistura["taxa_de_mistura"] * 100,
        marker="o", lw=2, color="#d62728", label="Taxa de mistura (4 classes)")

for s, taxa in zip(mistura.index, mistura["taxa_de_mistura"]):
    ax.annotate(f"{taxa*100:.2f}%", (s, taxa * 100),
                textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=9, fontweight="bold")

# marca onde r_01 cruza 1 (a partir daí as nuvens 0 e 1 se fundem)
s_critico = r_min
ax.axvline(s_critico, ls="--", color="gray", lw=1.2,
           label=f"$r_{{01}} = 1$ (em $s = {s_critico:.2f}$)")

ax.set_xlabel("fator de escala $s$")
ax.set_ylabel("taxa de mistura (%)")
ax.set_title("Figura 3 — Taxa de mistura $\\times$ fator de escala")
ax.set_xticks(SCALES)
ax.set_xticklabels([str(s) for s in SCALES])
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

**Resposta B.4 — a partir de qual fator de escala as nuvens deixam de ser separáveis por
retas?**

A partir de $s = 2$.

O critério geométrico e a evidência empírica coincidem:

- **$s = 0{,}5$** — taxa de mistura de **0,00%** (0 pontos em 400). Nuvens compactas e
  isoladas; qualquer conjunto razoável de retas separa tudo.
- **$s = 1$** — **5,00%** (20 pontos). O erro está concentrado nas classes 0 e 1 (8% e 12%
  dos seus pontos); as classes 2 e 3 seguem com 0% de mistura. Ainda é essencialmente
  separável por retas, com uma franja fina de erro.
- **$s = 2$** — **19,25%** (77 pontos), e agora *todas* as classes erram: 34%, 32%, 8% e 3%.
  É exatamente aqui que $r_{01}$ cai para **0,6629 < 1**: os centros 0 e 1 estão mais perto um
  do outro do que o raio típico das próprias nuvens. Cerca de um em cada cinco pontos cai do
  lado errado de *qualquer* reta que se desenhe — não é limitação do modelo, é sobreposição
  real das distribuições.
- **$s = 4$** — **48,25%** (193 pontos), com 65% e 62% de erro nas classes 0 e 1. As nuvens
  são praticamente uma só massa de pontos.

**O que acontece com o menor $r_{ij}$ nesse ponto?** Ele atravessa o limiar 1: de
$r_{01} = 1{,}3258$ em $s = 1$ para $r_{01} = 0{,}6629$ em $s = 2$ (o cruzamento exato se dá
em $s = 1{,}33$, marcado na Figura 3). Enquanto $r_{ij} > 1$ existe uma "folga" geométrica
entre o par de nuvens, por onde uma fronteira linear pode passar; quando $r_{ij} < 1$ essa
folga desaparece e as regiões de alta densidade das duas classes se interpenetram. Nenhuma
fronteira — reta ou curva — recupera essa informação.

## C — Análise

### C.1 — Sobreposição no dataset original ($s = 1$)

No dataset original as quatro classes formam três grupos visuais distintos:

- **Classes 0 e 1 se sobrepõem parcialmente.** É o único par problemático
  ($r_{01} = 1{,}3258$, o menor dos seis). Ambas são alongadas em $x_2$ e seus centros distam
  só 4,2426 — a cauda superior da classe 0 invade a região da classe 1. Isso aparece na tabela
  do item B.3: em $s = 1$, **8%** dos pontos da classe 0 e **12%** da classe 1 têm o centro da
  outra como o mais próximo.
- **Classe 2 é limpa.** Compacta ($\bar{\sigma}_2 = 0{,}9$) e deslocada para baixo
  ($\mu_y = 1$); **0%** de mistura.
- **Classe 3 é trivialmente separável.** Isolada em $\mu_x = 15$, a mais de 7,6 unidades de
  qualquer outro centro, com desvio pequeno em $x_1$; **0%** de mistura. Basta uma reta
  vertical em torno de $x_1 \approx 12$.

**Uma única fronteira linear separaria todas as classes?** Não. Um único hiperplano em
$\mathbb{R}^2$ (uma reta) divide o plano em apenas **duas** regiões, então no melhor caso
distingue 2 dos 4 grupos. É uma limitação estrutural, independente dos dados: é justamente por
isso que um único neurônio linear não resolve um problema de 4 classes, e se usa uma camada de
saída com 4 unidades — ou seja, 4 funções lineares combinadas.

**E um conjunto de fronteiras lineares?** Sim, quase perfeitamente. Com 3 a 4 retas dá para
isolar cada nuvem: uma reta vertical separando a classe 3, uma reta separando a classe 2, e uma
diagonal entre as classes 0 e 1. O único resíduo é a franja de sobreposição entre 0 e 1, que
**nenhuma** fronteira elimina — é o erro de Bayes do problema. Um classificador linear já
ficaria em torno de 95% de acerto neste dataset; camadas escondidas adicionariam curvatura à
fronteira 0–1, mas o ganho é marginal, porque o problema ali é sobreposição de densidades e não
formato de fronteira.

### C.2 — Fronteiras de decisão prováveis, esboçadas sobre a Figura 1

Para "esboçar" as fronteiras usei o classificador mais simples compatível com o que a rede
aprenderia: **atribuir cada ponto do plano ao centro de classe mais próximo**. Como todas as
nuvens têm o mesmo número de pontos e escalas comparáveis, essas fronteiras (o diagrama de
Voronoi dos 4 centros) são uma boa aproximação do que uma rede treinada produz — e são
exatamente um **conjunto de retas**, o mesmo mecanismo de uma camada de saída linear. Nada é
treinado: as fronteiras saem de uma conta de distância sobre uma grade.

In [ ]:
# Enquadramento do próprio dataset s = 1 (é a Figura 1 anotada)
folga4 = 1.0
X1LIM = (X[:, 0].min() - folga4, X[:, 0].max() + folga4)
Y1LIM = (X[:, 1].min() - folga4, X[:, 1].max() + folga4)

# Grade sobre o plano; cada célula recebe a classe do centro mais próximo
gx = np.linspace(X1LIM[0], X1LIM[1], 600)
gy = np.linspace(Y1LIM[0], Y1LIM[1], 600)
GX, GY = np.meshgrid(gx, gy)
grade = np.column_stack([GX.ravel(), GY.ravel()])

d_grade = np.linalg.norm(grade[:, None, :] - MEANS[None, :, :], axis=2)
regioes = d_grade.argmin(axis=1).reshape(GX.shape)

fig, ax = plt.subplots(figsize=(9, 6))

from matplotlib.colors import ListedColormap
ax.contourf(GX, GY, regioes, levels=np.arange(-0.5, N_CLASSES),
            cmap=ListedColormap(COLORS), alpha=0.13)
ax.contour(GX, GY, regioes, levels=np.arange(0.5, N_CLASSES - 0.5),
           colors="black", linewidths=1.6, linestyles="--")

plotar_nuvens(X, y, ax, tamanho=16)

# destaca os pontos que caem do lado errado dessa fronteira
_, prox_A = taxa_de_mistura(X, y)
erros = X[prox_A != y]
ax.scatter(erros[:, 0], erros[:, 1], s=90, facecolors="none",
           edgecolors="black", linewidths=1.3, zorder=6,
           label=f"lado errado da fronteira ({len(erros)} pts)")

ax.set_xlim(*X1LIM)
ax.set_ylim(*Y1LIM)
ax.set_title("Figura 1 (anotada) — fronteiras de decisão esperadas sobre o dataset $s = 1$")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend(title="Classe", loc="upper left", fontsize=8,
          title_fontsize=8, framealpha=0.9)
fig.tight_layout()
plt.show()

As fronteiras são **cinco segmentos de reta** que se encontram em **dois pontos triplos** (onde
três classes ficam equidistantes): a diagonal descendente entre as classes 0 e 1, o segmento
quase vertical 0–2 descendo à esquerda, o trecho 1–2 quase horizontal no meio, e as duas retas
que isolam a classe 3 (1–3 subindo e 2–3 descendo, à direita). Os círculos vazados marcam os
**20 pontos** (5,00% de 400) que caem do lado errado — praticamente todos na fronteira 0–1,
como previsto pela menor razão de separação.

### C.3 — Relação com o item B: o que acontece com a região de erro inevitável

As fronteiras da Figura 1 anotada dependem **só das médias**, e as médias não mudam com $s$.
Então as retas ficam paradas no mesmo lugar nos quatro datasets — o que cresce é a **massa de
pontos que atravessa essas retas**:

| $s$ | menor $r_{ij}$ ($r_{01}$) | taxa de mistura | região de erro |
|:---:|:---:|:---:|:---|
| 0,5 | 2,6517 | 0,00% | inexistente — as nuvens não alcançam as fronteiras |
| 1,0 | 1,3258 | 5,00% | franja fina, restrita ao par 0–1 |
| 2,0 | 0,6629 | 19,25% | faixas largas em torno de todas as fronteiras |
| 4,0 | 0,3315 | 48,25% | quase todo o plano é ambíguo |

Quanto mais dispersas as nuvens, **maior a região onde a rede necessariamente erra** — e por um
motivo que não é do modelo: nessa região dois ou mais rótulos são genuinamente prováveis para o
mesmo ponto $(x_1, x_2)$. O erro deixa de ser de *aproximação* (fronteira de formato errado,
que mais capacidade resolveria) e passa a ser *ruído irredutível* — o erro de Bayes.

A consequência prática para o projeto da arquitetura:

- Em $s \le 1$, a fronteira ótima é essencialmente linear e a rede pode ser rasa. Adicionar
  camadas só serve para curvar levemente a fronteira 0–1; o ganho é pequeno.
- Em $s \ge 2$, **nenhuma** arquitetura chega perto de 100%. Uma rede grande o suficiente
  memorizaria os pontos de treino e traçaria fronteiras retorcidas, mas isso é *overfitting*:
  estaria modelando o ruído amostral, não a estrutura. O teto de acerto é fixado pela geometria
  dos dados, e $r_{ij} < 1$ é o sinal antecipado disso — calculado com quatro médias e quatro
  desvios, sem treinar nada.

Ou seja: a razão de separação e a taxa de mistura são diagnósticos que se calculam *antes* de
escolher a arquitetura, e já dizem se o problema pede mais capacidade ou se pede dados
melhores.

# Exercício 2 — Não-Linearidade em Dimensões Mais Altas

Redes simples (como o Perceptron) só aprendem fronteiras lineares. Redes profundas se
justificam quando os dados **não** são linearmente separáveis. Este exercício contrasta dois
datasets de mesma dimensionalidade (5D) para tornar essa diferença explícita.

O `rng` continua sendo o mesmo do Exercício 1 — os sorteios abaixo dão sequência ao fluxo
aleatório já iniciado. Aqui entra o **scikit-learn**, exclusivamente para o `PCA` do item C;
nenhum modelo é treinado.

## A — Dataset I: gaussianas deslocadas

500 amostras para a Classe A e 500 para a Classe B, de uma normal multivariada em 5D:

$$\mu_A = [0, 0, 0, 0, 0], \qquad \mu_B = [1{,}5,\ 1{,}5,\ 1{,}5,\ 1{,}5,\ 1{,}5]$$

As duas classes têm **espalhamentos diferentes**: $\Sigma_B$ tem variâncias maiores (1,5 contra
1,0) e correlação **negativa** entre as duas primeiras features ($-0{,}7$), enquanto
$\Sigma_A$ tem correlação **positiva** ($+0{,}8$).

In [ ]:
from sklearn.decomposition import PCA

N_EX2 = 500

mu_A = np.zeros(5)
mu_B = np.full(5, 1.5)

Sigma_A = np.array([
    [1.0, 0.8, 0.1, 0.0, 0.0],
    [0.8, 1.0, 0.3, 0.0, 0.0],
    [0.1, 0.3, 1.0, 0.5, 0.0],
    [0.0, 0.0, 0.5, 1.0, 0.2],
    [0.0, 0.0, 0.0, 0.2, 1.0],
])

Sigma_B = np.array([
    [ 1.5, -0.7, 0.2, 0.0, 0.0],
    [-0.7,  1.5, 0.4, 0.0, 0.0],
    [ 0.2,  0.4, 1.5, 0.6, 0.0],
    [ 0.0,  0.0, 0.6, 1.5, 0.3],
    [ 0.0,  0.0, 0.0, 0.3, 1.5],
])

# covariâncias válidas precisam ser simétricas e positivas-definidas
for nome, S in [("Sigma_A", Sigma_A), ("Sigma_B", Sigma_B)]:
    autoval = np.linalg.eigvalsh(S)
    print(f"{nome}: simétrica = {np.allclose(S, S.T)}, "
          f"autovalores = {autoval.round(4)}, "
          f"positiva-definida = {bool((autoval > 0).all())}")

In [ ]:
X_A = rng.multivariate_normal(mu_A, Sigma_A, size=N_EX2)
X_B = rng.multivariate_normal(mu_B, Sigma_B, size=N_EX2)

X_I = np.vstack([X_A, X_B])
y_I = np.repeat([0, 1], N_EX2)
ROTULOS_I = ["Classe A", "Classe B"]

print(f"Dataset I: X_I.shape = {X_I.shape}, amostras por classe = {np.bincount(y_I)}")
print(f"valores ausentes: {int(np.isnan(X_I).sum())}")

pd.DataFrame(X_I, columns=[f"x{i+1}" for i in range(5)]).assign(
    classe=[ROTULOS_I[k] for k in y_I]
).groupby("classe").mean().round(3)

As médias empíricas ficam em $(-0{,}037,\ -0{,}038,\ 0{,}029,\ 0{,}037,\ 0{,}026)$ para a
Classe A e $(1{,}473,\ 1{,}557,\ 1{,}438,\ 1{,}420,\ 1{,}388)$ para a Classe B — próximas dos
$[0,\dots]$ e $[1{,}5,\dots]$ teóricos. Ambas as matrizes de covariância são simétricas e
positivas-definidas (menor autovalor: 0,1582 em $\Sigma_A$ e 0,4979 em $\Sigma_B$), então são
covariâncias válidas.

## B — Dataset II: cascas concêntricas

Segundo dataset, também com 500 amostras por classe e também em 5D, mas com estrutura
**radial**:

1. direções uniformes na esfera unitária de $\mathbb{R}^5$: sorteia-se
   $v \sim \mathcal{N}(0, I_5)$ e normaliza-se, $u = v / \lVert v \rVert$;
2. **Classe C (núcleo):** raio $\rho \sim \mathcal{N}(2{,}0,\ 0{,}4)$;
3. **Classe D (casca):** raio $\rho \sim \mathcal{N}(5{,}0,\ 0{,}4)$;
4. cada ponto é $x = \rho \cdot u$.

> **Convenção adotada:** lemos $\mathcal{N}(2{,}0,\ 0{,}4)$ como média 2,0 e **desvio-padrão**
> 0,4 — mesma notação [média, desvio] usada no Exercício 1. Com desvio 0,4 os raios ficam
> concentrados em torno de 2 e de 5, que é a estrutura de "cascas" pretendida.

In [ ]:
def amostrar_casca(raio_medio, raio_desvio, n=N_EX2, dim=5):
    # Pontos em uma casca esférica: direção uniforme na esfera x raio gaussiano.
    # Usa o `rng` global. As direções vêm de uma normal isotrópica normalizada —
    # é o jeito padrão de sortear uniformemente na esfera de R^dim.
    v = rng.normal(0.0, 1.0, size=(n, dim))
    u = v / np.linalg.norm(v, axis=1, keepdims=True)   # direções unitárias
    rho = rng.normal(raio_medio, raio_desvio, size=n)  # raios
    return rho[:, None] * u


X_C = amostrar_casca(2.0, 0.4)   # núcleo
X_D = amostrar_casca(5.0, 0.4)   # casca

X_II = np.vstack([X_C, X_D])
y_II = np.repeat([0, 1], N_EX2)
ROTULOS_II = ["Classe C (núcleo)", "Classe D (casca)"]

print(f"Dataset II: X_II.shape = {X_II.shape}, amostras por classe = {np.bincount(y_II)}")
print(f"valores ausentes: {int(np.isnan(X_II).sum())}")

# checagem de que as direções realmente estão na esfera unitária antes de escalar
for nome, Xk in [("C (núcleo)", X_C), ("D (casca)", X_D)]:
    normas = np.linalg.norm(Xk / np.linalg.norm(Xk, axis=1, keepdims=True), axis=1)
    print(f"  classe {nome}: ||u|| em [{normas.min():.6f}, {normas.max():.6f}] "
          f"(deve ser 1,0)")

## C — Visualizar e comparar

Não dá para plotar um gráfico 5D diretamente, então reduzimos a dimensionalidade.

### C.1 — Figura 4: PCA de cada dataset em 2 dimensões

O PCA é ajustado sobre o dataset completo (as duas classes juntas), sem padronizar as
features: em ambos os datasets as 5 features já estão na mesma escala, então centrar — o que o
PCA faz internamente — é suficiente.

In [ ]:
pca_I = PCA(n_components=5).fit(X_I)
pca_II = PCA(n_components=5).fit(X_II)

Z_I = pca_I.transform(X_I)[:, :2]
Z_II = pca_II.transform(X_II)[:, :2]

var_I = pca_I.explained_variance_ratio_
var_II = pca_II.explained_variance_ratio_

variancia = pd.DataFrame(
    {"Dataset I (gaussianas)": var_I, "Dataset II (cascas)": var_II},
    index=[f"PC{i+1}" for i in range(5)],
)
variancia.loc["PC1+PC2"] = variancia.loc[["PC1", "PC2"]].sum()

print(f"Dataset I : PC1 = {var_I[0]*100:.2f}%, PC2 = {var_I[1]*100:.2f}%, "
      f"PC1+PC2 = {var_I[:2].sum()*100:.2f}%")
print(f"Dataset II: PC1 = {var_II[0]*100:.2f}%, PC2 = {var_II[1]*100:.2f}%, "
      f"PC1+PC2 = {var_II[:2].sum()*100:.2f}%")

# alinhamento entre o PC1 e a direção que separa as classes
for nome, pca_obj, ca, cb in [("Dataset I", pca_I, X_A, X_B),
                              ("Dataset II", pca_II, X_C, X_D)]:
    d = cb.mean(axis=0) - ca.mean(axis=0)
    d = d / np.linalg.norm(d)
    cos = abs(pca_obj.components_[0] @ d)
    print(f"{nome}: ângulo(PC1, direção dos centros) = "
          f"{np.degrees(np.arccos(cos)):.1f}°  (|cos| = {cos:.4f})")

(variancia * 100).round(2)

In [ ]:
CORES_EX2 = ["#1f77b4", "#d62728"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.4))

for ax, (Z, yy, rotulos, var, nome) in zip(axes, [
    (Z_I, y_I, ROTULOS_I, var_I, "Dataset I — gaussianas deslocadas"),
    (Z_II, y_II, ROTULOS_II, var_II, "Dataset II — cascas concêntricas"),
]):
    for k, rot in enumerate(rotulos):
        pts = Z[yy == k]
        ax.scatter(pts[:, 0], pts[:, 1], s=12, alpha=0.55,
                   color=CORES_EX2[k], edgecolors="none", label=rot)
    ax.set_title(f"{nome}\n(PC1+PC2 = {var[:2].sum()*100:.2f}% da variância)", fontsize=10)
    ax.set_xlabel(f"PC1 — {var[0]*100:.2f}% da variância")
    ax.set_ylabel(f"PC2 — {var[1]*100:.2f}% da variância")
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(title="Classe", loc="upper right", fontsize=8, title_fontsize=8,
              framealpha=0.9)

fig.suptitle("Figura 4 — Projeção PCA em 2D dos dois datasets 5D", fontsize=13)
fig.tight_layout()
plt.show()

### C.2 — Variância explicada pelas duas primeiras componentes

| Componente | Dataset I (gaussianas) | Dataset II (cascas) |
|:---|:---:|:---:|
| PC1 | **51,28%** | **21,37%** |
| PC2 | **16,07%** | **20,90%** |
| **PC1 + PC2** | **67,35%** | **42,27%** |
| PC3 | 14,59% | 19,67% |
| PC4 | 12,04% | 19,51% |
| PC5 | 6,01% | 18,54% |

**Em qual dataset a projeção 2D preserva melhor a informação relevante para classificação?**
No **Dataset I**, e por duas razões distintas:

1. **Quantidade:** as duas primeiras componentes retêm 67,35% da variância, contra 42,27% no
   Dataset II. No Dataset II a variância é quase perfeitamente *isotrópica* — as cinco
   componentes ficam entre 18,54% e 21,37%, ou seja, nenhuma direção é privilegiada. Isso é
   esperado: as direções $u$ são uniformes na esfera, então a nuvem não tem eixo preferencial,
   e PC1 e PC2 acabam sendo escolhas essencialmente arbitrárias. Projetar em 2D descarta
   57,73% da variância sem nada em troca.
2. **Qualidade — que é o que de fato importa:** no Dataset I o PC1 aponta quase exatamente na
   direção que separa as classes. O ângulo entre o PC1 e a direção $\mu_B - \mu_A$ é de apenas
   **7,6°** ($|\cos| = 0{,}9913$). Como o deslocamento entre os centros é a maior fonte de
   variância do conjunto, o PCA "encontra" a direção discriminante sem nunca ter visto os
   rótulos. No Dataset II não existe direção discriminante linear alguma para o PCA encontrar
   — a informação está no **raio**, que não é função linear das coordenadas.

Na Figura 4 isso aparece direto: à esquerda, duas nuvens deslocadas com sobreposição parcial;
à direita, um disco pequeno (Classe C) **enterrado dentro** de um disco maior (Classe D).

### C.3 — Medidas em 5D: distância entre centros e histograma dos raios

In [ ]:
linhas_ex2 = []
for nome, Xa, Xb in [("Dataset I", X_A, X_B), ("Dataset II", X_C, X_D)]:
    c_a, c_b = Xa.mean(axis=0), Xb.mean(axis=0)
    r_a, r_b = np.linalg.norm(Xa, axis=1), np.linalg.norm(Xb, axis=1)
    linhas_ex2.append({
        "dataset": nome,
        "||mu_1 - mu_2||": np.linalg.norm(c_a - c_b),
        "raio_medio_c1": r_a.mean(), "raio_desvio_c1": r_a.std(ddof=1),
        "raio_medio_c2": r_b.mean(), "raio_desvio_c2": r_b.std(ddof=1),
        "raio_max_c1": r_a.max(), "raio_min_c2": r_b.min(),
        "raios_se_sobrepoem": bool(r_a.max() > r_b.min()),
    })

centros = pd.DataFrame(linhas_ex2).set_index("dataset")

print(f"Dataset I : ||mu_A - mu_B|| = {centros.loc['Dataset I', '||mu_1 - mu_2||']:.4f}  "
      f"(teórico: ||[1,5]x5|| = {np.linalg.norm(mu_B - mu_A):.4f})")
print(f"Dataset II: ||mu_C - mu_D|| = {centros.loc['Dataset II', '||mu_1 - mu_2||']:.4f}  "
      f"(teórico: 0, porque as direções são uniformes)")

centros.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

bins = np.linspace(0, 8, 60)
for ax, (Xa, Xb, rotulos, nome) in zip(axes, [
    (X_A, X_B, ROTULOS_I, "Dataset I — gaussianas deslocadas"),
    (X_C, X_D, ROTULOS_II, "Dataset II — cascas concêntricas"),
]):
    for k, Xk in enumerate([Xa, Xb]):
        ax.hist(np.linalg.norm(Xk, axis=1), bins=bins, alpha=0.6,
                color=CORES_EX2[k], edgecolor="white", linewidth=0.4,
                label=rotulos[k])
    ax.set_title(nome, fontsize=10)
    ax.set_xlabel("raio $\\|x\\|$ (em 5D)")
    ax.set_ylabel("frequência (nº de pontos)")
    ax.legend(title="Classe", fontsize=8, title_fontsize=8, framealpha=0.9)

fig.suptitle("Figura 5 — Histograma dos raios $\\|x\\|$, com as classes sobrepostas "
             "no mesmo eixo", fontsize=13)
fig.tight_layout()
plt.show()

**Resposta C.3.**

| | Dataset I | Dataset II |
|:---|:---:|:---:|
| $\lVert \mu_1 - \mu_2 \rVert$ (5D) | **3,2524** (teórico: 3,3541) | **0,2215** (teórico: 0) |
| raio médio — classe 1 | 2,102 ± 0,823 | 2,004 ± 0,411 |
| raio médio — classe 2 | 4,094 ± 1,235 | 5,018 ± 0,406 |
| raios se sobrepõem? | **sim** (máx. A = 4,80 > mín. B = 1,10) | **não** (máx. C = 3,28 < mín. D = 3,44) |

Os dois datasets são **imagens espelhadas** um do outro:

- No **Dataset I**, os centros estão longe (3,2524) mas os raios se confundem completamente —
  a Classe A ocupa raios de 0,42 a 4,80 e a Classe B de 1,10 a 7,49. Separar pelo raio seria
  péssimo; separar pela posição funciona bem.
- No **Dataset II**, os centros praticamente coincidem (0,2215 — e esse resíduo é só ruído
  amostral de 500 pontos, o valor teórico é exatamente 0) mas os raios se separam com folga:
  o ponto mais externo do núcleo tem raio 3,28 e o mais interno da casca tem 3,44. Existe um
  intervalo vazio entre eles. Separar pela posição é impossível; separar pelo raio é trivial.

## D — Análise

### D.1 — Centros coincidentes e raios separados: o que isso diz sobre hiperplanos?

Um hiperplano é o conjunto $\{x : w^{\top}x = b\}$, e classificar com ele significa olhar
**uma única projeção** $w^{\top}x$ e compará-la com um limiar. A distância entre os centros é
justamente a quantidade que essa projeção consegue explorar: o melhor critério linear baseado
em posição usa a direção $\mu_2 - \mu_1$, e o que ele enxerga é o quanto os centros se afastam
ao longo de $w$. Se os centros coincidem, **toda** projeção linear leva as duas classes a
distribuições com a mesma média.

Já o raio $\lVert x \rVert$ **não é** função linear de $x$ — é quadrática. A combinação
"centros juntos + raios separados" significa, portanto: existe uma estrutura perfeitamente
discriminante nos dados, mas ela é invisível para qualquer hiperplano. A informação está na
*distância à origem*, não na *posição ao longo de uma direção*.

A medida do Exercício 1 confirma isso numericamente. Aplicando a mesma regra do centro mais
próximo (agora em 5D, com os centros empíricos de cada classe):

In [ ]:
for nome, Xa, Xb, yy in [("Dataset I", X_A, X_B, y_I), ("Dataset II", X_C, X_D, y_II)]:
    Xt = np.vstack([Xa, Xb])
    centros_emp = np.vstack([Xa.mean(axis=0), Xb.mean(axis=0)])
    taxa, _ = taxa_de_mistura(Xt, yy, means=centros_emp)
    print(f"{nome}: taxa de mistura (centro mais próximo, 5D) = {taxa*100:.2f}%")

**12,00%** no Dataset I contra **47,00%** no Dataset II. Os 47% são estatisticamente
indistinguíveis de jogar uma moeda (50%, para 2 classes): a regra baseada em centros — e, por
extensão, qualquer fronteira linear que se apoie na posição dos centros — não extrai
absolutamente nada do Dataset II.

### D.2 — Por que o Dataset II não se resolve com fronteira linear, por mais dados que se colete

O argumento é geométrico e não depende do tamanho da amostra.

Fixe qualquer direção unitária $w$ e olhe a projeção $t = w^{\top}x$. Como as direções $u$ são
**uniformes na esfera**, para todo $w$ existem pontos da casca apontando tanto a favor quanto
contra $w$: a Classe D produz projeções que vão de aproximadamente $-5$ a $+5$. A Classe C,
com raio em torno de 2, produz projeções confinadas a aproximadamente $[-2{,}6,\ +2{,}6]$.
Ou seja, **o intervalo ocupado pela Classe C está contido no intervalo ocupado pela Classe D**,
qualquer que seja $w$:

In [ ]:
direcoes = {
    "e1 (eixo x1)": np.eye(5)[0],
    "diagonal 1/sqrt(5)": np.full(5, 1 / np.sqrt(5)),
    "direção dos centros": (X_D.mean(axis=0) - X_C.mean(axis=0)) /
                           np.linalg.norm(X_D.mean(axis=0) - X_C.mean(axis=0)),
}

linhas_proj = []
for nome, w in direcoes.items():
    proj_C, proj_D = X_C @ w, X_D @ w
    linhas_proj.append({
        "direcao_w": nome,
        "C_min": proj_C.min(), "C_max": proj_C.max(), "C_media": proj_C.mean(),
        "D_min": proj_D.min(), "D_max": proj_D.max(), "D_media": proj_D.mean(),
        "C_contido_em_D": bool(proj_D.min() < proj_C.min() and proj_C.max() < proj_D.max()),
    })

pd.DataFrame(linhas_proj).set_index("direcao_w").round(3)

Em toda direção testada a faixa da Classe C está estritamente dentro da faixa da Classe D, e as
médias das duas classes são praticamente iguais (todas próximas de 0). Nenhum limiar $b$ sobre
$w^{\top}x$ separa as classes: escolhendo "$w^{\top}x > b \Rightarrow$ casca", todos os pontos
da casca que apontam para o lado oposto de $w$ caem abaixo de $b$ e são classificados como
núcleo.

Isso limita o **teto de acerto de qualquer hiperplano** a cerca de **75%**: no melhor caso ele
acerta 100% do núcleo e aproximadamente metade da casca — os 50% que apontam para o lado certo
de $w$. Coletar mais dados não muda nada: cada ponto novo da casca também é sorteado com
direção uniforme, então a proporção se mantém. O problema não é falta de dados nem amostra não
representativa; é que a família de funções (lineares) não contém nenhuma função que resolva o
problema. É exatamente a situação em que uma camada escondida com ativação não-linear deixa de
ser opcional.

### D.3 — Uma projeção 2D "misturada" prova inseparabilidade no espaço original?

**Não.** PCA é uma transformação **linear** que maximiza *variância retida*, não *separação
entre classes* — ele nem enxerga os rótulos. Projetar de 5D para 2D descarta três direções, e
se a informação discriminante morar justamente nelas, classes perfeitamente separáveis aparecem
embaralhadas na tela.

Justificando com os nossos resultados: a projeção do Dataset II descarta 57,73% da variância, e
o que se vê na Figura 4 é o núcleo enterrado dentro da casca — **54,0% dos pontos da Classe D
caem, na projeção 2D, a uma distância da origem menor que a do ponto mais externo da Classe C**.
Visualmente, misturados. E, no entanto, no espaço original de 5D as classes são
**perfeitamente separáveis** — só que por uma fronteira não-linear. A função pedida é
exatamente a da dica:

$$g(x) = \lVert x \rVert^2 = \sum_{i=1}^{5} x_i^2,
\qquad \text{classifique como casca se } g(x) > \tau$$

In [ ]:
# quanto da Classe D parece "dentro" da Classe C na projeção 2D (Figura 4)
raio2d_C = np.linalg.norm(Z_II[y_II == 0], axis=1)
raio2d_D = np.linalg.norm(Z_II[y_II == 1], axis=1)
print(f"projeção 2D: {(raio2d_D < raio2d_C.max()).mean()*100:.1f}% dos pontos da Classe D "
      f"caem dentro do raio máximo da Classe C\n")

g_C = (X_C ** 2).sum(axis=1)   # ||x||^2 do núcleo
g_D = (X_D ** 2).sum(axis=1)   # ||x||^2 da casca

print(f"||x||^2 do núcleo (C): [{g_C.min():.3f}, {g_C.max():.3f}]")
print(f"||x||^2 da casca  (D): [{g_D.min():.3f}, {g_D.max():.3f}]")
print(f"intervalo vazio entre as classes: ({g_C.max():.3f}, {g_D.min():.3f})\n")

for tau, origem in [(12.25, "limiar natural: raio (2+5)/2 = 3,5 -> tau = 3,5^2"),
                    (11.30, "meio do intervalo vazio observado")]:
    acertos = int((g_C <= tau).sum() + (g_D > tau).sum())
    print(f"tau = {tau:6.2f}  ({origem})")
    print(f"             acurácia = {acertos}/1000 = {acertos/10:.1f}%")

Com o limiar "natural", derivado só dos parâmetros geradores — o raio no meio do caminho entre
as duas cascas, $r^\ast = (2+5)/2 = 3{,}5$, ou seja $\tau = 3{,}5^2 = 12{,}25$ — a função
acerta **99,9%** (999 de 1000; erra um único ponto da casca, o de raio 3,44). Usando o meio do
intervalo vazio efetivamente observado, $\tau = 11{,}30$, a separação é **perfeita: 100,0%**.
Existe uma faixa inteira de limiares que funciona, $\tau \in (10{,}756,\ 11{,}867)$, porque os
dois conjuntos de valores de $\lVert x \rVert^2$ são disjuntos.

Repare no que $g$ é, em termos de rede neural: uma soma dos quadrados das entradas. Um
perceptron não a representa, mas ela é trivial para uma rede com uma camada escondida — ou,
equivalentemente, para um modelo linear sobre as features expandidas $x_i^2$. É o mesmo
princípio do truque de kernel: o problema fica linear no espaço de features adequado.

**Conclusão do item.** Uma projeção 2D em que as classes parecem misturadas **não prova nada**
sobre separabilidade no espaço original — prova apenas que as classes não são separáveis *por
aquelas duas direções lineares específicas*. O Dataset II mostra os dois lados da moeda ao
mesmo tempo: nenhum hiperplano o resolve (item D.2), mas a separabilidade perfeita existe e é
alcançada por uma função quadrática simples. O diagnóstico correto, portanto, é: antes de
concluir "inseparável" a partir de um gráfico, vale testar medidas que não sejam lineares —
foi exatamente o que o histograma de raios da Figura 5 revelou.

# Exercício 3 — Preparando Dados Reais para uma Rede Neural

Este exercício usa um dataset real do Kaggle. A tarefa é fazer o pré-processamento necessário
para deixá-lo adequado a uma rede neural que usa **tangente hiperbólica (`tanh`)** como ativação
nas camadas escondidas.

> **Fonte dos dados.** *Spaceship Titanic*, competição Getting Started do Kaggle
> (<https://www.kaggle.com/competitions/spaceship-titanic>), arquivo `train.csv` — o único
> rotulado. Licença **CC BY 4.0**. O arquivo está versionado em `dataset/train.csv`, ao lado
> deste notebook.

## A — Conhecer os dados

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

titanic = pd.read_csv("dataset/train.csv")

print(f"shape = {titanic.shape}  ({titanic.shape[0]} passageiros, "
      f"{titanic.shape[1]} colunas)")
titanic.head()

### A.2 — Objetivo do dataset e balanceamento das classes

O objetivo é prever a coluna **`Transported`**: se o passageiro foi *transportado para uma
dimensão alternativa* durante a colisão da Spaceship Titanic com a anomalia espaço-temporal.
É um alvo **binário** (`True`/`False`), então o problema é de **classificação binária** — a
rede teria uma única saída com ativação sigmoide (ou duas com softmax).

In [ ]:
balanco = titanic["Transported"].value_counts().rename("contagem").to_frame()
balanco["proporcao_%"] = (titanic["Transported"].value_counts(normalize=True) * 100).round(2)

prop_positiva = titanic["Transported"].mean()
print(f"classe positiva (True):  {int(titanic['Transported'].sum())} de "
      f"{len(titanic)}  =  {prop_positiva*100:.2f}%")
print(f"classe negativa (False): {int((~titanic['Transported']).sum())} de "
      f"{len(titanic)}  =  {(1-prop_positiva)*100:.2f}%")
print(f"razão entre classes: {titanic['Transported'].sum() / (~titanic['Transported']).sum():.4f}")

balanco

**Balanceamento: 50,36% `True` contra 49,64% `False`** (4378 e 4315 passageiros). É um dataset
praticamente **balanceado** — a razão entre as classes é 1,0146. Isso simplifica bastante o
treino: acurácia é uma métrica honesta aqui (o baseline de chutar sempre a classe majoritária
acerta só 50,36%), não há necessidade de pesos por classe na função de perda nem de
reamostragem.

### A.3 — Features numéricas e categóricas

In [ ]:
COLS_GASTO = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
COLS_NUM   = ["Age"] + COLS_GASTO
COLS_CAT   = ["HomePlanet", "CryoSleep", "Destination", "VIP"]
COLS_DROP  = ["PassengerId", "Cabin", "Name"]
ALVO       = "Transported"

tipos = pd.DataFrame({
    "coluna": titanic.columns,
    "dtype": [str(t) for t in titanic.dtypes],
    "valores_unicos": [titanic[c].nunique() for c in titanic.columns],
})
tipos["papel"] = [
    "alvo" if c == ALVO else
    "numérica" if c in COLS_NUM else
    "categórica" if c in COLS_CAT else
    "descartada" for c in titanic.columns
]
tipos = tipos.set_index("coluna")
tipos

- **Numéricas (6):** `Age` e as cinco colunas de gasto — `RoomService`, `FoodCourt`,
  `ShoppingMall`, `Spa`, `VRDeck`. São contínuas e não-negativas.
- **Categóricas (4):** `HomePlanet` (3 categorias), `Destination` (3), `CryoSleep` (booleana) e
  `VIP` (booleana). As duas booleanas entram aqui porque são rótulos, não grandezas — não faz
  sentido calcular média delas.
- **Descartadas (3):** `PassengerId` é identificador (8693 valores únicos, informação zero para
  generalizar), `Name` também (8473 únicos), e `Cabin` é um código composto `deck/num/side` com
  6560 valores únicos — precisaria ser quebrado em três features antes de ser útil, o que o
  enunciado pede para não fazer.

### A.4 — Valores ausentes

In [ ]:
faltantes = pd.DataFrame({
    "n_ausentes": titanic.isna().sum(),
    "pct_ausentes": (titanic.isna().mean() * 100).round(2),
}).sort_values("n_ausentes", ascending=False)

print(f"total de células ausentes: {int(titanic.isna().sum().sum())} "
      f"({titanic.isna().sum().sum() / titanic.size * 100:.2f}% da tabela)")
print(f"linhas com pelo menos um ausente: {int(titanic.isna().any(axis=1).sum())} "
      f"({titanic.isna().any(axis=1).mean()*100:.2f}%)")
print(f"linhas completas: {int((~titanic.isna().any(axis=1)).sum())}")

faltantes

**Nenhuma coluna passa de 2,5% de ausentes**, e o alvo `Transported` e o `PassengerId` estão
completos. A ausência é distribuída de forma quase uniforme entre as 12 colunas restantes —
de 2,06% (`Age`, 179 células) a 2,50% (`CryoSleep`, 217 células).

O detalhe que importa: embora cada coluna individualmente tenha pouca coisa faltando,
**2087 linhas (24,01%) têm pelo menos um valor ausente**. Descartar linhas incompletas jogaria
fora um quarto do dataset — daí a imputação ser a estratégia certa aqui, e não o `dropna`.

### A.5 — Colunas de gasto: média, mediana e máximo

In [ ]:
gastos_stats = titanic[COLS_GASTO].agg(["mean", "median", "max"]).T
gastos_stats.columns = ["media", "mediana", "maximo"]
gastos_stats["razao_media_mediana"] = np.where(
    gastos_stats["mediana"] > 0,
    gastos_stats["media"] / gastos_stats["mediana"].replace(0, np.nan),
    np.inf,
)
gastos_stats["pct_zeros"] = [(titanic[c] == 0).mean() * 100 for c in COLS_GASTO]
gastos_stats["assimetria"] = [titanic[c].skew() for c in COLS_GASTO]

gastos_stats.round(2)

| coluna | média | mediana | máximo | % de zeros | assimetria |
|:---|---:|---:|---:|---:|---:|
| RoomService | 224,69 | **0,00** | 14 327 | 65,5% | 6,36 |
| FoodCourt | 458,08 | **0,00** | 29 813 | 64,7% | 7,38 |
| ShoppingMall | 173,73 | **0,00** | 23 492 | 66,0% | 8,19 |
| Spa | 311,14 | **0,00** | 22 408 | 65,4% | 7,74 |
| VRDeck | 304,85 | **0,00** | 24 133 | 65,3% | 7,57 |

**O que a diferença entre média e mediana diz sobre o espalhamento e a assimetria?**

A mediana é **0,00 em todas as cinco colunas**, enquanto as médias vão de 173,73 a 458,08. Uma
média centenas de unidades acima de uma mediana nula é o retrato de uma distribuição
**fortemente assimétrica à direita** (cauda longa positiva), e os números explicam por quê:
cerca de **dois terços dos passageiros gastam exatamente zero** em cada amenidade, enquanto uma
minoria gasta valores altíssimos — o máximo de `FoodCourt` é 29 813, mais de **65 vezes a média**
da própria coluna.

Sobre o **espalhamento**: a massa da distribuição está concentrada em um ponto (o zero) e o
alcance vai até dezenas de milhares. O coeficiente de assimetria confirma — entre **6,36 e
8,19**, quando uma distribuição simétrica tem assimetria 0 e valores acima de 1 já são
considerados fortemente assimétricos.

Isso tem consequência direta para o item C: se essas colunas forem padronizadas como estão, os
outliers viram valores de $z$ enormes, e uma rede com `tanh` satura imediatamente neles. É
exatamente o problema que a transformação $\log(1+x)$ resolve.

## B — Dividir antes de transformar

### B.1 — Split estratificado 80/20 com semente fixa

In [ ]:
# O scikit-learn não aceita um np.random.Generator em `random_state` (só inteiro ou
# RandomState), então passamos a mesma semente 42 usada no `rng` global do relatório.
SEMENTE_SPLIT = 42

X_bruto = titanic.drop(columns=[ALVO])
y_alvo = titanic[ALVO].astype(int)   # True/False -> 1/0

X_treino_bruto, X_teste_bruto, y_treino, y_teste = train_test_split(
    X_bruto, y_alvo,
    test_size=0.20,
    stratify=y_alvo,          # preserva a proporção das classes nos dois lados
    random_state=SEMENTE_SPLIT,
)

print(f"treino: {X_treino_bruto.shape[0]} linhas  ({X_treino_bruto.shape[0]/len(titanic)*100:.1f}%)")
print(f"teste : {X_teste_bruto.shape[0]} linhas  ({X_teste_bruto.shape[0]/len(titanic)*100:.1f}%)")
print(f"\nproporção da classe positiva:")
print(f"  dataset completo: {y_alvo.mean()*100:.2f}%")
print(f"  treino          : {y_treino.mean()*100:.2f}%")
print(f"  teste           : {y_teste.mean()*100:.2f}%")

**6954 linhas de treino e 1739 de teste.** A estratificação funcionou: a proporção da classe
positiva é 50,36% no dataset completo, 50,36% no treino e 50,37% no teste — a diferença é o
arredondamento de não dar para dividir 1739 linhas exatamente ao meio.

### B.2 — Por que o split vem *antes* da imputação e do scaling

Porque toda estatística usada numa transformação — a mediana da imputação, a média e o desvio
do scaler, a lista de categorias do one-hot — é *informação aprendida dos dados*. Se ela for
calculada sobre o dataset inteiro, os valores do conjunto de teste participam do cálculo, e o
teste deixa de ser um conjunto que o modelo nunca viu: ele já influenciou silenciosamente como
o treino foi transformado. Isso é **vazamento de dados** (*data leakage*).

O efeito prático é que a performance medida no teste fica **otimista** e não representa mais o
que aconteceria em produção. Em produção, o dado novo chega um de cada vez e precisa ser
transformado com estatísticas fixas, calculadas no passado — exatamente a situação que o split
prévio simula. Por isso, daqui para frente, **todo `.fit()` vê apenas `X_treino_bruto`**, e ao
teste se aplica só `.transform()`.

## C — Pré-processamento

A ativação `tanh` produz valores em $[-1, 1]$ e sua região útil (onde a derivada não é
desprezível) é aproximadamente $|z| < 2$ — em $z = 3$ a derivada já é 0,0099, e em $z = 5$ é
0,00018. Entradas fora dessa faixa **saturam** os neurônios: a saída gruda em $\pm 1$ e o
gradiente que retorna é praticamente zero, travando o aprendizado daquele peso. Toda a
sequência abaixo existe para entregar à rede uma matriz centrada e em escala compatível.

### C.1 — Dados ausentes: estratégia por tipo de coluna

In [ ]:
# Numéricas -> MEDIANA. Escolha motivada pela assimetria vista em A.5: com dois terços
# de zeros e caudas até 30 mil, a média seria puxada pelos outliers e imputaria um
# valor que quase ninguém tem. A mediana é robusta a esse tipo de cauda.
imputador_num = SimpleImputer(strategy="median")
imputador_num.fit(X_treino_bruto[COLS_NUM])          # fit SÓ no treino

# Categóricas -> MODA (categoria mais frequente). São 3 a 2 categorias por coluna e
# menos de 2,5% de ausentes; criar uma categoria "Desconhecido" para tão poucos casos
# adicionaria uma coluna one-hot quase vazia, com mais ruído do que sinal.
imputador_cat = SimpleImputer(strategy="most_frequent")
imputador_cat.fit(X_treino_bruto[COLS_CAT])          # fit SÓ no treino

print("medianas aprendidas no treino (numéricas):")
for col, val in zip(COLS_NUM, imputador_num.statistics_):
    print(f"  {col:14s} -> {val:8.2f}")

print("\nmodas aprendidas no treino (categóricas):")
for col, val in zip(COLS_CAT, imputador_cat.statistics_):
    print(f"  {col:14s} -> {val}")

**Justificativa de cada escolha.**

- **Numéricas → mediana.** O item A.5 mostrou assimetria entre 6,36 e 8,19. A média de
  `FoodCourt` é 452,61 no treino, mas a mediana é 0,00. Imputar pela média inventaria um gasto
  de ~450 créditos para passageiros sobre os quais não sabemos nada — e ~65% dos passageiros
  gastam zero. A mediana não se move com os outliers e, neste dataset, imputa o valor
  semanticamente correto: **0 para todas as colunas de gasto** (e 27 anos para `Age`).
- **Categóricas → moda.** As quatro colunas têm 2 ou 3 categorias e menos de 2,5% de ausentes.
  As modas aprendidas no treino são `Earth`, `CryoSleep=False`, `TRAPPIST-1e` e `VIP=False`.
  A alternativa — criar uma categoria `"Desconhecido"` — é defensável quando a ausência é
  informativa ou frequente; com 2% de ausentes ela geraria uma coluna one-hot preenchida em
  ~2% das linhas, praticamente ruído para a rede.

Ambos os imputadores foram ajustados **apenas em `X_treino_bruto`**, e as estatísticas acima
(medianas e modas) são as que serão aplicadas também ao teste.

### C.2 — Features categóricas: one-hot encoding

In [ ]:
# One-hot: cada categoria vira uma coluna 0/1. É a codificação adequada aqui porque
# HomePlanet e Destination são NOMINAIS — não existe ordem entre Earth, Europa e Mars,
# então codificá-las como 0/1/2 inventaria uma relação de grandeza que não existe.
#
# handle_unknown="ignore": se no teste aparecer uma categoria que não estava no treino,
# o codificador NÃO quebra e NÃO cria coluna nova — emite zeros em todo o bloco daquela
# coluna. O número de features fica idêntico entre treino e teste, que é o requisito
# para a matriz alimentar a mesma rede.
codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

cat_treino_imputado = pd.DataFrame(
    imputador_cat.transform(X_treino_bruto[COLS_CAT]),
    columns=COLS_CAT, index=X_treino_bruto.index,
)
codificador.fit(cat_treino_imputado)                 # fit SÓ no treino

NOMES_ONEHOT = list(codificador.get_feature_names_out())
print(f"{len(NOMES_ONEHOT)} colunas one-hot geradas:")
for nome in NOMES_ONEHOT:
    print("  -", nome)

In [ ]:
# Demonstração do caso "categoria nova no teste": um passageiro de um planeta que o
# treino nunca viu.
passageiro_novo = pd.DataFrame(
    [["Krypton", False, "TRAPPIST-1e", False]], columns=COLS_CAT
)
saida = codificador.transform(passageiro_novo)[0].astype(int)

print("entrada:", passageiro_novo.values.tolist()[0])
print()
for nome, valor in zip(NOMES_ONEHOT, saida):
    marca = "  <- bloco HomePlanet todo zerado" if nome.startswith("HomePlanet") else ""
    print(f"  {nome:26s} = {valor}{marca}")

**Como o código trata uma categoria que aparece no teste mas não no treino.** O
`OneHotEncoder` foi construído com `handle_unknown="ignore"`. Sem esse argumento, o
`.transform()` lançaria exceção e o pipeline quebraria em produção. Com ele, a categoria
desconhecida é representada por **zeros em todo o bloco daquela coluna** — como mostra a
demonstração acima: o passageiro de `"Krypton"` recebe `HomePlanet_Earth = 0`,
`HomePlanet_Europa = 0`, `HomePlanet_Mars = 0`, enquanto os blocos de `CryoSleep`,
`Destination` e `VIP` são codificados normalmente.

É o comportamento desejado: a rede recebe "nenhum dos planetas conhecidos" em vez de um
palpite errado, **o número de colunas não muda** (17 no treino e 17 no teste), e nenhuma
exceção é levantada. Neste dataset específico o caso não chega a ocorrer — as três categorias
de `HomePlanet` e as três de `Destination` aparecem nos dois lados do split — mas o código
está preparado.

### C.3 — Feature engineering: `TotalSpend`, e descarte de colunas

In [ ]:
def montar_numericas(X_bruto_parcial):
    # Aplica os imputadores JÁ AJUSTADOS (só transform) e cria o TotalSpend.
    num = pd.DataFrame(
        imputador_num.transform(X_bruto_parcial[COLS_NUM]),
        columns=COLS_NUM, index=X_bruto_parcial.index,
    )
    # TotalSpend: soma das cinco colunas de gasto, calculada sobre os valores brutos
    # (antes do log) para que represente o gasto total real do passageiro.
    num["TotalSpend"] = num[COLS_GASTO].sum(axis=1)
    return num


num_treino = montar_numericas(X_treino_bruto)
num_teste  = montar_numericas(X_teste_bruto)

# As colunas descartadas: identificadores e o código composto de cabine.
print(f"colunas descartadas: {COLS_DROP}")
print(f"  PassengerId: {titanic['PassengerId'].nunique()} valores únicos (identificador)")
print(f"  Name       : {titanic['Name'].nunique()} valores únicos (identificador)")
print(f"  Cabin      : {titanic['Cabin'].nunique()} valores únicos (código deck/num/side)")

print(f"\nTotalSpend no treino: média = {num_treino['TotalSpend'].mean():.2f}, "
      f"mediana = {num_treino['TotalSpend'].median():.2f}, "
      f"máximo = {num_treino['TotalSpend'].max():.2f}")
print(f"colunas numéricas agora: {list(num_treino.columns)}")

`TotalSpend` é a soma das cinco colunas de gasto, calculada **sobre os valores brutos** (depois
da imputação, antes do log) para que represente o gasto total real do passageiro. Ela dá à rede
um resumo direto de "quanto esse passageiro consumiu", que de outra forma ela teria que
aprender somando cinco entradas — e a distinção entre gastar zero em tudo e gastar em alguma
coisa é justamente o padrão mais forte do dataset (passageiros em `CryoSleep` ficam confinados
à cabine e não gastam nada).

As três colunas descartadas não carregam sinal generalizável: `PassengerId` (8693 valores
únicos) e `Name` (8473) são identificadores, e `Cabin` (6560) é um código composto que
precisaria ser quebrado em `deck`, `num` e `side` para virar feature — fora do escopo pedido.

### C.4 — Caudas pesadas: $\log(1 + x)$

In [ ]:
COLS_LOG = COLS_GASTO + ["TotalSpend"]

# Assimetria antes e depois, medida no treino
comparacao_skew = pd.DataFrame({
    "assimetria_antes": [num_treino[c].skew() for c in COLS_LOG],
    "assimetria_depois": [np.log1p(num_treino[c]).skew() for c in COLS_LOG],
    "max_antes": [num_treino[c].max() for c in COLS_LOG],
    "max_depois": [np.log1p(num_treino[c]).max() for c in COLS_LOG],
}, index=COLS_LOG)

# log1p = log(1 + x): definido em x = 0 (vira 0) e monotônico, então preserva a ordem
# entre os passageiros. Aplicado às 5 colunas de gasto e ao TotalSpend.
for bloco in (num_treino, num_teste):
    bloco[COLS_LOG] = np.log1p(bloco[COLS_LOG])

comparacao_skew.round(3)

In [ ]:
# Por que isso importa para uma rede com tanh: comparação dos z-scores COM e SEM o log
sem_log = montar_numericas(X_treino_bruto)                 # versão crua, para comparar
z_sem_log = StandardScaler().fit_transform(sem_log)
z_com_log = StandardScaler().fit_transform(num_treino)

print(f"SEM log1p: maior |z| = {np.abs(z_sem_log).max():6.2f}  ->  "
      f"tanh({np.abs(z_sem_log).max():.2f}) = {np.tanh(np.abs(z_sem_log).max()):.10f}")
print(f"COM log1p: maior |z| = {np.abs(z_com_log).max():6.2f}  ->  "
      f"tanh({np.abs(z_com_log).max():.2f}) = {np.tanh(np.abs(z_com_log).max()):.10f}")
print(f"\nfração de valores com |z| > 3: {np.mean(np.abs(z_sem_log) > 3)*100:.2f}% sem log, "
      f"{np.mean(np.abs(z_com_log) > 3)*100:.2f}% com log")

**Por que essa transformação ajuda uma rede com `tanh`?**

A assimetria despenca de **6,36–8,19 para 1,14–1,26** nas cinco colunas de gasto (e para
$-0{,}20$ em `TotalSpend`, praticamente simétrica). Mas o número decisivo é o efeito sobre a
escala depois da padronização:

- **Sem o log**, o maior $z$-score do bloco numérico é **22,27**. E
  $\tanh(22{,}27) = 1{,}0000000000$ — saturação total. A derivada de `tanh` nesse ponto é
  $1 - \tanh^2 \approx 0$, então o gradiente que volta por aquele neurônio é numericamente
  zero: **aquele passageiro não ensina nada à rede**, e os pesos ligados àquela entrada param de
  se atualizar. Pior, para a média e o desvio serem dominados por esses outliers, todos os
  passageiros "normais" ficam espremidos num intervalo minúsculo em torno de zero, onde `tanh`
  é quase linear — a rede perde a capacidade de distingui-los.
- **Com o log**, o maior $z$-score cai para **3,51**, com $\tanh(3{,}51) = 0{,}9982$ — no limite
  da região útil, mas ainda com derivada não-nula. A fração de valores com $|z| > 3$ cai de
  1,74% para **0,06%**.

O log comprime a cauda sem destruir a informação: é **monotônico**, então a ordem entre os
passageiros é preservada — quem gastou mais continua tendo o maior valor. E usamos
$\log(1+x)$, não $\log(x)$, justamente porque dois terços dos valores são zero e $\log(0)$ é
indefinido; $\log(1+0) = 0$ mantém o zero como zero.

### C.5 — Scaling: padronização (média 0, desvio 1)

In [ ]:
# Escolha: StandardScaler (média 0, desvio 1) nas colunas numéricas.
escalonador = StandardScaler()
escalonador.fit(num_treino)                          # fit SÓ no treino

num_treino_esc = escalonador.transform(num_treino)
num_teste_esc  = escalonador.transform(num_teste)

# Comparação com a alternativa (normalização para [-1, 1]), para justificar a escolha
from sklearn.preprocessing import MinMaxScaler
alternativa = MinMaxScaler(feature_range=(-1, 1)).fit_transform(num_treino)

print("PADRONIZAÇÃO (escolhida)")
print(f"  média por coluna: {num_treino_esc.mean(axis=0).round(6)}")
print(f"  intervalo: [{num_treino_esc.min():.4f}, {num_treino_esc.max():.4f}]")
print(f"  {np.mean(np.abs(num_treino_esc) < 2)*100:.1f}% dos valores em |z| < 2 "
      f"(região ativa do tanh)")

print("\nNORMALIZAÇÃO [-1, 1] (descartada)")
print(f"  média por coluna: {alternativa.mean(axis=0).round(3)}")
print(f"  intervalo: [{alternativa.min():.4f}, {alternativa.max():.4f}]")
print(f"  {np.mean(alternativa < -0.8)*100:.1f}% dos valores abaixo de -0,8 "
      f"(região saturada do tanh)")

**Escolhi a padronização**, e a comparação acima é a justificativa.

A `tanh` é **centrada em zero** — é essa a razão de ela ser preferida à sigmoide em camadas
escondidas. Uma entrada padronizada tem média 0 e desvio 1, então ela cai naturalmente na
região de maior derivada da `tanh`, onde o gradiente é mais forte: **95,6% dos valores ficam em
$|z| < 2$**.

A normalização para $[-1, 1]$ *também* produz valores no alcance da `tanh`, mas com um problema
que os números expõem: como as distribuições de gasto continuam assimétricas mesmo depois do
log (com um pico em zero), o `MinMaxScaler` empurra a maior parte da massa para perto do limite
inferior. As médias por coluna ficam entre $-0{,}19$ e $-0{,}67$, e **55,0% de todos os valores
caem abaixo de $-0{,}8$** — ou seja, mais da metade das entradas começaria o treino na zona
saturada negativa da `tanh`. Além disso, o `MinMaxScaler` é definido pelo mínimo e pelo máximo,
os dois pontos mais sensíveis a outliers: um único passageiro extremo comprime todos os outros.

**Mínimo e máximo resultantes** (bloco numérico, treino): **$[-1{,}9961,\ 3{,}5080]$**. Os
valores completos, já com o one-hot, estão no item D.2.

## D — Verificar e visualizar

### D.1 — Figura 6: `FoodCourt` antes e depois do pré-processamento

In [ ]:
# Monta a matriz final: [numéricas escaladas | one-hot], treino e teste
onehot_treino = codificador.transform(
    pd.DataFrame(imputador_cat.transform(X_treino_bruto[COLS_CAT]),
                 columns=COLS_CAT, index=X_treino_bruto.index)
)
onehot_teste = codificador.transform(
    pd.DataFrame(imputador_cat.transform(X_teste_bruto[COLS_CAT]),
                 columns=COLS_CAT, index=X_teste_bruto.index)
)

X_treino = np.hstack([num_treino_esc, onehot_treino])
X_teste  = np.hstack([num_teste_esc, onehot_teste])

NOMES_FEATURES = list(num_treino.columns) + NOMES_ONEHOT
idx_foodcourt = NOMES_FEATURES.index("FoodCourt")

print(f"matriz de treino: {X_treino.shape}   matriz de teste: {X_teste.shape}")
print(f"{len(NOMES_FEATURES)} features: {NOMES_FEATURES}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

antes = X_treino_bruto["FoodCourt"].fillna(imputador_num.statistics_[1]).to_numpy()
depois = X_treino[:, idx_foodcourt]
y_tr = y_treino.to_numpy()
ROTULOS_ALVO = ["Transported = False", "Transported = True"]

for ax, (dados, titulo, xlabel, nbins) in zip(axes, [
    (antes, "Antes — FoodCourt bruto (créditos)",
     "FoodCourt (créditos gastos)", 60),
    (depois, "Depois — log(1+x) e padronização",
     "FoodCourt transformado (z-score)", 40),
]):
    for k, rot in enumerate(ROTULOS_ALVO):
        ax.hist(dados[y_tr == k], bins=nbins, alpha=0.6, color=CORES_EX2[k],
                edgecolor="white", linewidth=0.3, label=rot)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("frequência (nº de passageiros)")
    ax.legend(title="Classe", fontsize=8, title_fontsize=8, framealpha=0.9)

axes[0].set_yscale("log")   # sem escala log a barra do zero esconde toda a cauda
axes[0].set_ylabel("frequência (escala log)")

fig.suptitle("Figura 6 — Distribuição de FoodCourt no treino, antes e depois do "
             "pré-processamento", fontsize=13)
fig.tight_layout()
plt.show()

O painel da esquerda precisa de **escala logarítmica no eixo y** para ser legível: 64,7% dos
passageiros estão empilhados na primeira barra (gasto zero) e a cauda se estende até 29 813
créditos com uma ou duas contagens por barra. É a assinatura visual de uma distribuição de
cauda pesada.

Depois de $\log(1+x)$ e da padronização, o painel da direita mostra uma distribuição que a rede
consegue usar: contida em $[-0{,}64,\ 2{,}86]$, centrada em zero, com a grande massa de
"gastou zero" virando um pico único em $z = -0{,}64$ e os gastadores espalhados num corpo largo
e bem povoado entre $-0{,}5$ e $2{,}5$ — em vez da cauda rarefeita de uma contagem por barra que
se via antes. Note que a separação entre as classes fica visível — quem
foi transportado gastou sistematicamente menos.

### D.2 — Verificações finais

In [ ]:
print("=" * 62)
print("VERIFICAÇÕES FINAIS")
print("=" * 62)

print(f"\n1) NaN restantes")
print(f"   treino: {int(np.isnan(X_treino).sum())}    teste: {int(np.isnan(X_teste).sum())}")
print(f"   infinitos -> treino: {int(np.isinf(X_treino).sum())}  "
      f"teste: {int(np.isinf(X_teste).sum())}")

print(f"\n2) shape final da matriz de features")
print(f"   treino: {X_treino.shape}   (alvo: {y_treino.shape})")
print(f"   teste : {X_teste.shape}   (alvo: {y_teste.shape})")
print(f"   {len(num_treino.columns)} numéricas escaladas + {len(NOMES_ONEHOT)} one-hot "
      f"= {X_treino.shape[1]} features")

print(f"\n3) faixa de valores (compatibilidade com tanh)")
print(f"   treino: [{X_treino.min():.4f}, {X_treino.max():.4f}]")
print(f"   teste : [{X_teste.min():.4f}, {X_teste.max():.4f}]")
print(f"   bloco numérico  - treino: [{num_treino_esc.min():.4f}, {num_treino_esc.max():.4f}]"
      f"  teste: [{num_teste_esc.min():.4f}, {num_teste_esc.max():.4f}]")
print(f"   bloco one-hot   - treino: [{onehot_treino.min():.0f}, {onehot_treino.max():.0f}]"
      f"  teste: [{onehot_teste.min():.0f}, {onehot_teste.max():.0f}]")
print(f"   |valor| < 2 em {np.mean(np.abs(X_treino) < 2)*100:.1f}% do treino "
      f"(região de gradiente forte do tanh)")
print(f"   |valor| > 3 em {np.mean(np.abs(X_treino) > 3)*100:.2f}% do treino")

**Checagens explícitas:**

- **Nenhum `NaN` restante** — 0 no treino e 0 no teste, e também nenhum infinito. As 2087 linhas
  que tinham pelo menos um ausente foram preservadas via imputação, em vez de descartadas.
- **Shape final:** treino **(6954, 17)** e teste **(1739, 17)** — 7 numéricas escaladas
  (`Age`, as 5 de gasto e `TotalSpend`) + 10 colunas one-hot. O número de colunas é idêntico
  nos dois conjuntos, como tem que ser.
- **Faixa de valores compatível com `tanh`:** treino em $[-1{,}9961,\ 3{,}5080]$ e teste em
  $[-1{,}9961,\ 3{,}3687]$. O bloco one-hot é 0/1 e o bloco numérico é o que define os
  extremos. **98,2% dos valores do treino estão em $|v| < 2$**, a região onde a derivada da
  `tanh` é significativa, e apenas 0,02% passam de 3.

O teste ficar dentro da faixa do treino (máximo 3,3687 contra 3,5080) é o comportamento
esperado e **não foi imposto**: o scaler foi ajustado só no treino, então o teste poderia
perfeitamente ter extrapolado. Que não tenha extrapolado indica que os dois conjuntos vêm da
mesma distribuição — mais uma evidência de que o split estratificado ficou bem feito.

### D.3 — Reflexão: qual decisão de pré-processamento mais afetaria o treino?

A transformação $\log(1+x)$ nas colunas de gasto, com folga — e a comparação numérica do item
C.4 mostra por quê. Sem ela, o maior $z$-score da matriz é **22,27**, e
$\tanh(22{,}27) = 1{,}0000000000$ em precisão dupla: a derivada $1 - \tanh^2$ é exatamente zero
em ponto flutuante, então o gradiente que atravessa aquele neurônio **desaparece**. Não é uma
degradação suave de performance; é um caminho de aprendizado que fecha. E o dano não se limita
aos outliers: como média e desvio são calculados com eles dentro, os ~65% de passageiros que
gastam zero e os que gastam algumas centenas ficam todos comprimidos num intervalo estreito
perto de zero, onde a `tanh` é aproximadamente linear — a rede perde justamente a resolução na
faixa onde estão quase todos os dados. Com o log, o maior $z$ cai para 3,51 e 95,6% das
entradas ficam na região de gradiente forte.

As demais decisões importam menos, e é instrutivo dizer por quê. A **escolha do scaler**
(padronização × normalização) muda a distribuição inicial das ativações e a velocidade de
convergência, mas nenhuma das duas zera gradiente da forma que a ausência do log zera — é uma
diferença de eficiência, não de viabilidade. A **estratégia de imputação** afeta apenas ~2% das
células, e como a mediana das colunas de gasto é 0 — semanticamente correto para quem não
consumiu — o impacto é pequeno. O **one-hot** é praticamente obrigatório: codificar
`HomePlanet` como 0/1/2 inventaria uma ordem entre Earth, Europa e Mars, e a rede gastaria
capacidade para desfazer essa relação falsa; mas com apenas 3 categorias o estrago seria
limitado. E o **`TotalSpend`** é conveniência: a rede poderia aprender essa soma sozinha a
partir das cinco entradas, já que é uma combinação linear delas.

Em resumo: o log é a única decisão da lista que separa "a rede aprende" de "a rede não
aprende". As outras separam "aprende melhor" de "aprende pior".

# Resumo dos resultados

Tabela-índice dos números calculados ao longo do relatório, reunidos em um só lugar. Ela não
substitui nenhuma análise — cada valor aqui é resultado de uma célula executada acima, e a
tabela é montada em código a partir das mesmas variáveis, para não haver divergência entre o
que foi calculado e o que está reportado.

In [ ]:
PENDENTE = "— (a preencher no Exercício 3)"

linhas_resumo = [
    (1,  "Taxa de mistura em s = 0,5",
         f"{mistura.loc[0.5, 'taxa_de_mistura']*100:.2f}%  "
         f"({int(mistura.loc[0.5, 'pontos_misturados'])}/400 pontos)"),
    (2,  "Taxa de mistura em s = 1,0",
         f"{mistura.loc[1.0, 'taxa_de_mistura']*100:.2f}%  "
         f"({int(mistura.loc[1.0, 'pontos_misturados'])}/400 pontos)"),
    (3,  "Taxa de mistura em s = 2,0",
         f"{mistura.loc[2.0, 'taxa_de_mistura']*100:.2f}%  "
         f"({int(mistura.loc[2.0, 'pontos_misturados'])}/400 pontos)"),
    (4,  "Taxa de mistura em s = 4,0",
         f"{mistura.loc[4.0, 'taxa_de_mistura']*100:.2f}%  "
         f"({int(mistura.loc[4.0, 'pontos_misturados'])}/400 pontos)"),
    (5,  "Menor r_ij em s = 1,0, e qual par",
         f"r = {r_min:.4f}  no par {par_min}"),
    (6,  "Distância entre centros — Dataset I",
         f"{np.linalg.norm(X_A.mean(0) - X_B.mean(0)):.4f}  (teórico 3,3541)"),
    (7,  "Distância entre centros — Dataset II",
         f"{np.linalg.norm(X_C.mean(0) - X_D.mean(0)):.4f}  (teórico 0)"),
    (8,  "Variância explicada PC1 + PC2 — Dataset I",
         f"{var_I[:2].sum()*100:.2f}%  "
         f"(PC1 {var_I[0]*100:.2f}% + PC2 {var_I[1]*100:.2f}%)"),
    (9,  "Variância explicada PC1 + PC2 — Dataset II",
         f"{var_II[:2].sum()*100:.2f}%  "
         f"(PC1 {var_II[0]*100:.2f}% + PC2 {var_II[1]*100:.2f}%)"),
    (10, "Proporção da classe positiva em Transported",
         f"{prop_positiva*100:.2f}% True  "
         f"({int(titanic[ALVO].sum())}/{len(titanic)});  "
         f"{(1-prop_positiva)*100:.2f}% False"),
    (11, "Média e mediana de FoodCourt no treino, antes de transformar",
         f"média = {X_treino_bruto['FoodCourt'].mean():.2f};  "
         f"mediana = {X_treino_bruto['FoodCourt'].median():.2f}  "
         f"(máximo = {X_treino_bruto['FoodCourt'].max():.0f})"),
    (12, "shape final da matriz de features de treino",
         f"{X_treino.shape}  "
         f"({len(num_treino.columns)} numéricas + {len(NOMES_ONEHOT)} one-hot);  "
         f"teste {X_teste.shape}"),
    (13, "Mínimo e máximo do treino e do teste após o scaling",
         f"treino [{X_treino.min():.4f}, {X_treino.max():.4f}];  "
         f"teste [{X_teste.min():.4f}, {X_teste.max():.4f}]"),
]

resumo_geral = pd.DataFrame(linhas_resumo, columns=["#", "Item", "Valor"]).set_index("#")

with pd.option_context("display.max_colwidth", 200):
    display(resumo_geral)

### Leitura dos números

**Exercício 1.** A taxa de mistura cresce de 0,00% para 48,25% enquanto as fronteiras de
decisão permanecem exatamente onde estavam — elas dependem só das médias, que não mudam com
$s$. O menor $r_{ij}$ é o do par (0, 1), $1{,}3258$, e é ele que governa a transição: ao
cruzar o limiar 1 (o que acontece em $s = 1{,}33$, e já está consumado em $s = 2$, onde
$r_{01} = 0{,}6629$), a folga geométrica entre as duas nuvens desaparece. A dificuldade é
**quantitativa**: o modelo certo continua sendo linear, o que aumenta é o erro irredutível.

**Exercício 2.** Os dois datasets são imagens espelhadas. O Dataset I tem centros distantes
(3,2524) e raios sobrepostos; o Dataset II tem centros coincidentes (0,2215, ruído amostral em
torno do valor teórico 0) e raios disjuntos. O PCA reflete isso: no Dataset I duas componentes
retêm 67,35% da variância e o PC1 fica a 7,6° da direção discriminante; no Dataset II a
variância é isotrópica (42,27% em duas componentes) porque não há direção privilegiada. Aqui a
dificuldade é **qualitativa**: não existe erro irredutível — as classes são perfeitamente
separáveis por $g(x) = \lVert x \rVert^2$ — mas nenhuma função linear resolve, com teto de
~75% para qualquer hiperplano.

**Exercício 3.** O dataset real é praticamente balanceado (50,36% da classe positiva), tem
ausentes em todas as colunas menos duas (nenhuma acima de 2,50%, mas 24,01% das linhas com pelo
menos um) e colunas de gasto com assimetria entre 6,36 e 8,19 — média em centenas contra mediana
zero. Depois do pipeline ajustado só no treino, a matriz final é (6954, 17) e (1739, 17), sem
nenhum `NaN`, com valores em $[-1{,}9961,\ 3{,}5080]$ — compatível com `tanh`, e com 98,2% das
entradas na região de gradiente forte. A decisão de maior impacto é o $\log(1+x)$: sem ele o
maior $z$-score é 22,27, onde a `tanh` satura e o gradiente zera.

**Conclusão geral.** O primeiro caso pede dados melhores; o segundo pede um modelo mais
expressivo; o terceiro mostra que, antes dessa escolha, há um trabalho de preparação que decide
se a rede consegue aprender. Distinguir um do outro *antes* de treinar qualquer coisa é exatamente o que essas
medidas geométricas simples — razão de separação, taxa de mistura, distância entre centros,
histograma de raios e variância explicada — permitem fazer.